In [1]:
# pip install protobuf sentencepiece tiktoken transformers torch sentencepiece

In [ ]:
from sklearn.ensemble import RandomForestClassifier

from pathlib import Path
import pandas as pd
import numpy as np
import os

import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import GridSearchCV

In [3]:
data_dir = 'data_readinglevel'
x_train_df = pd.read_csv(os.path.join(data_dir, 'x_train.csv'))
y_train_df = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))

x_test_df = pd.read_csv(os.path.join(data_dir, 'x_test.csv'))

N, n_cols = x_train_df.shape
print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
print("Shape of y_train_df: %s" % str(y_train_df.shape))

Shape of x_train_df: (5557, 32)
Shape of y_train_df: (5557, 5)


In [4]:
# Create the binary classification column y
# 2-3 is labeled as 0
# 4-5 is labeled as 1

y_train_df["true_labels"] = np.where(y_train_df["Coarse Label"].str.contains("Key Stage 2-3"), 0, 1)
y = y_train_df.true_labels.to_numpy() 

In [ ]:
class DeBERTaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='microsoft/deberta-v3-small', batch_size=8):
        self.model_name = model_name
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        # Move model to GPU if available
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        self.model.eval()
        all_embeddings = []
        texts = X.tolist()

        for i in range(0, len(texts), self.batch_size):
            batch_texts = texts[i:i + self.batch_size]
            inputs = self.tokenizer(batch_texts, padding=True, truncation=True, 
                                    max_length=512, return_tensors="pt").to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                embeddings = outputs.last_hidden_state.mean(dim=1)
                all_embeddings.append(embeddings.cpu().numpy())

        return np.vstack(all_embeddings)

    def get_feature_names_out(self, input_features=None):
        # deberta-v3-small has 768 dimensions
        return np.array([f"deberta_dim_{i}" for i in range(768)])

Define Features we want to use from the set

In [20]:
feature_columns = [
    "avg_word_length",
    "avg_sentence_length",
    "info_type_token_ratio",
    "readability_FleschReadingEase",
    "readability_DaleChallIndex", "punctuation_frequency"
]

deberta_feat_extractor = DeBERTaTransformer(model_name='microsoft/deberta-v3-small', batch_size=16)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 69916.49it/s]
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expec

Extract Embeddings Once

In [ ]:
# 1. Run the Embeddings onto the text once
x_train_embeddings = deberta_feat_extractor.transform(x_train_df['text'])
x_test_embeddings = deberta_feat_extractor.transform(x_test_df['text'])

In [ ]:
# Save the Embeddings cause ain't no way I'm running ts again
np.save('x_train_deberta.npy', x_train_embeddings)
np.save('x_test_deberta.npy', x_test_embeddings)

In [52]:
# 2. Combine numerical features with the pre-calculated embeddings
scaler = MinMaxScaler()

x_train_num_scaled = scaler.fit_transform(x_train_df[feature_columns])
x_test_num_scaled = scaler.transform(x_test_df[feature_columns])

# 2. Combine with pre-calculated DeBERTa embeddings
x_train_combined = np.hstack([x_train_num_scaled, x_train_embeddings])
x_test_combined = np.hstack([x_test_num_scaled, x_test_embeddings])

print(f"Final training shape: {x_train_combined.shape}")

Final training shape: (5557, 774)


In [63]:
# 3. Update the Hyperparameter Grid for Random Forest
param_grid = {'n_estimators': [100, 200, 300], 'max_depth': [10, 20]}


# Use the original author column for grouping
groups = x_train_df['author']

In [61]:
# 4. Initialize GridSearchCV
cv = GroupKFold(n_splits=5)

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    param_grid=param_grid,
    cv=GroupKFold(n_splits=5),
    scoring="roc_auc",
    verbose=1
)

In [62]:
grid.fit(x_train_combined, y, groups=groups)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


KeyboardInterrupt: 

In [36]:
print("Best Parameters:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

Best Parameters: {'max_depth': 10, 'n_estimators': 200}
Best CV ROC-AUC: 0.7800422675956018


In [ ]:
# 1. Get the classifier directly
# Since you trained on pre-calculated features, this IS the RandomForest
classifier = grid.best_estimator_

# 2. Reconstruct the feature names manually
# These must be in the exact order you concatenated them (e.g., Num Features + DeBERTa)
num_feature_names = [f"num__{col}" for col in feature_columns]
deberta_feature_names = [f"deberta_dim_{i}" for i in range(768)]
all_feature_names = num_feature_names + deberta_feature_names

# 3. Extract Feature Importances
importances = classifier.feature_importances_

# 4. Create Importance DataFrame
feature_importance = pd.DataFrame({
    'feature': all_feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

# 5. Print the Top 20 Most Predictive Features
print("\n--- Top 20 Most Influential Features (Random Forest) ---")
print(feature_importance[['feature', 'importance']].head(20))

# 6. Analyze Structural Feature Importance
structural_features = feature_importance[feature_importance['feature'].str.startswith('num__')]
print("\n--- Importance of Structural Readability Metrics ---")
print(structural_features)

# 7. Aggregate DeBERTa Embedding Importance
deberta_importance = feature_importance[feature_importance['feature'].str.contains('deberta_dim')]
print(f"\nAverage Importance per DeBERTa Dimension: {deberta_importance['importance'].mean():.6f}")
print(f"Total Importance Contribution of DeBERTa: {deberta_importance['importance'].sum():.4f}")



--- Top 20 Most Influential Features (Random Forest) ---
                        feature  importance
1      num__avg_sentence_length    0.032148
710             deberta_dim_704    0.011725
656             deberta_dim_650    0.011019
160             deberta_dim_154    0.009707
191             deberta_dim_185    0.009538
61               deberta_dim_55    0.008895
5    num__punctuation_frequency    0.007924
11                deberta_dim_5    0.006893
291             deberta_dim_285    0.006860
705             deberta_dim_699    0.006407
272             deberta_dim_266    0.006197
588             deberta_dim_582    0.005789
163             deberta_dim_157    0.005787
77               deberta_dim_71    0.005511
468             deberta_dim_462    0.005397
680             deberta_dim_674    0.005039
132             deberta_dim_126    0.004975
416             deberta_dim_410    0.004562
527             deberta_dim_521    0.004401
573             deberta_dim_567    0.004147

--- Importance of

In [49]:
y_proba1_test = classifier.predict_proba(x_test_combined)[:, 1]
print(y_proba1_test)
np.savetxt('yproba1_test.txt', y_proba1_test)

[0.87932621 0.8541034  0.64791015 ... 0.57084304 0.62263232 0.55768165]
